In [ ]:
import sys
from pathlib import Path
from datetime import timedelta

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from simple_slurm import Slurm
from metab_processing.metab_travlr_config import DATA_DIR

# Full dataset directory paths: DATA_DIR / <project> / <dataset>.
PATHS = [f'{DATA_DIR}/Alexi_UC_Spliced/13114_HS1_HC-Slice_1']
# PATHS = [f'{DATA_DIR}/Xenium/Primary_Dermal_Melanoma']  # melanoma, other project
TOP_N = 1000
ANNOT = '25_06_11_ICI_5K_Coarse_annotations'

SCRIPT = str(_root / 'metab_processing' / 'LinearRegression' / 'hvg_prep.py')
PYTHON = '/global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python'
LOG_DIR = Path('/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/hvg')
LOG_DIR.mkdir(parents=True, exist_ok=True)

cmd = f'{PYTHON} {SCRIPT} --paths {" ".join(PATHS)} --top-n {TOP_N} --annot {ANNOT}'

In [ ]:
# A40 (savio3)
slurm = Slurm(account='fc_wagnerlabfca', partition='savio3_gpu', qos='a40_gpu3_normal',
              gres='gpu:A40:1', cpus_per_task=8, time=timedelta(hours=12), ignore_pbs=True,
              job_name='hvg_prep', output=str(LOG_DIR / 'hvg_%j.log'))

# A5000 (savio4)
# slurm = Slurm(account='fc_wagnerlabfca', partition='savio4_gpu', qos='a5k_gpu4_normal',
#               gres='gpu:A5000:1', cpus_per_task=4, time=timedelta(hours=12), ignore_pbs=True,
#               job_name='hvg_prep', output=str(LOG_DIR / 'hvg_%j.log'))

print(cmd)
slurm.sbatch(cmd)